In [1]:
import pandas as pd
import re

In [2]:
nightmarket = "Lehua"
nightmarket_ch = "樂華夜市"

In [3]:
def e_load_raw_data(dir,typename):
    try:
        path = f"../data/{dir}/{dir}_{typename}.csv"
    except:
        print("輸入錯誤，請重新輸入")
        return 
    df = pd.read_csv(path)
    return df

In [4]:
def t_get_user_id(df):
    df["user_id"] = df["user_id"].apply(lambda x : x.split("contrib/")[1].split("/reviews?")[0])
    return df

In [5]:
second_ad_level = ["市","鄉","鎮","區"]
municipality = ["台北市","新北市","桃園市","台中市","高雄市","台南市"]
counties = ["基隆市","新竹市","新竹縣","苗栗縣","台中市","南投市","彰化縣","雲林縣","嘉義縣","屏東縣","台東縣","花蓮縣","宜蘭縣","澎湖縣","金門縣","連江縣"]

In [6]:
def find_city(x):
    for city in municipality:
        if city in x:
            return city,x.find(city)
    for county in counties:
        if county in x:
            return county,x.find(county)
    return None,-1

In [7]:
def find_second_ad_level(x):
    city,city_idx = find_city(x)
    if city in counties : 
        for level in second_ad_level[:3]:
            if level in x:
                return city,city_idx,x.find(level)
    elif city in municipality : 
        return city,city_idx,x.find("區")
    else                      : 
        for level in second_ad_level:
            if level in x:
                return city,city_idx,x.find(level)
    return city,city_idx,-1

In [8]:
def get_city_address(x):
    city,city_idx,second_ad_idx = find_second_ad_level(x)
    if second_ad_idx + 2 == len(x) : return city,x
    if city_idx < second_ad_idx : return city,x[second_ad_idx + 1 :]
    else:
        if "太麻里鄉" in x : return city,x[ : second_ad_idx - 3]
        else:
            if second_ad_idx > -1    : return city,x[ : second_ad_idx - 2]
            else                      : return city,x[city_idx + 3:]

In [9]:
def t_clean_address(df):
    df.insert(1,"nm_city","")
    df[["nm_city","地址"]] = df["地址"].apply(get_city_address).apply(pd.Series)
    return df

In [10]:
def t_is_localguide(df):
    def local_guide_or_not(x):
        if pd.isna(x) or "在地嚮導" not in x:
            return "FALSE"
        else :
            return "TRUE"
    df.insert(3,"is_local_guide","")
    df["is_local_guide"] = df["local_guide"].apply(local_guide_or_not)
    df["local_guide"] = df["local_guide"].str.lstrip("在地嚮導· ")
    return df

In [11]:
def t_get_comment_count(df):
    def comment_count(x):
        if pd.isna(x):
            return 0
        x = x.rstrip("則評論")
        return re.search(r"\d*",x).group()
    df.insert(4,"review_count","")
    df["review_count"] = df["local_guide"].apply(comment_count)
    return df

In [12]:
def t_get_photo_count(df):
    def photo_count(x):
        if pd.isna(x):
            return 0
        x = x.split("則評論 · ")[-1]
        return re.search(r"\d*",x).group()
    df.insert(5,"photo_count","")
    df["photo_count"] = df["local_guide"].apply(photo_count)
    df.drop("local_guide",axis=1,inplace=True)
    return df

In [13]:
def t_get_time(df):
    def unit_to_month(x):
        if   "天" in x : return 1 / 30
        elif "週" in x : return 0.25
        elif "月" in x : return 1.5
        elif "年" in x : return 18
        else           : return 0
    df_utom = df["time_unit"].apply(unit_to_month)
    df["time_num"] = df["time_num"].map(int)
    df.insert(9,"months_ago","")
    df["months_ago"] = df["time_num"] * df_utom
    return df

In [14]:
def t_get_star(df):
    df.insert(6,"rating_star","")
    df["rating_star"] = df["rating_stars"].apply(lambda x : re.search(r"\d",x).group())
    df.drop("rating_stars",axis=1,inplace=True)
    return df

In [15]:
def t_clean_comment(df):
    def clear_special_char(x):
        if pd.isna(x):
            return
        else:
            return re.sub(r"'"," ",x)
    df["comment"] = df["comment"].apply(clear_special_char)
    df.rename(columns={"comment":"content_clean"},inplace=True)
    return df

In [16]:
def t_clean_tag(df):
    def clean_tag(x):
        return re.sub(r"[\[\'\]]*","",x)
    df["st_tag"] = df["st_tag"].apply(clean_tag)
    return df

In [17]:
df_store = e_load_raw_data(nightmarket,"restaurant")
df_store.head(2)

,餐廳名稱,地址,餐廳連結,latitude,longitude,distance,st_score,st_price,st_total_com,st_tag
0,樂華夜市三鮮羹,234新北市永和區永平路146號樂華夜市內,https://www.google.com/maps/place/%E6%A8%82%E8...,25.008416,121.508817,0.146853,4.1,$1-200,1478,"['土魠魚', '米糕', '排隊', '魷魚', '蝦仁羹', '米粉', '醋', '肉..."
1,鐘點棧 當歸鴨【台北樂華夜市總店】銅板美食|小吃店外送推薦,234新北市永和區保平路18巷8號,https://www.google.com/maps/place/%E9%90%98%E9...,25.007947,121.509835,0.080487,4.2,$1-200,1252,"['碗', '鴨肉飯', '當歸湯', '麵線', '米血', '腿', '香腸', '鴨油..."


In [18]:
df_store = t_clean_address(df_store)
df_store.head(2)

,餐廳名稱,nm_city,地址,餐廳連結,latitude,longitude,distance,st_score,st_price,st_total_com,st_tag
0,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,https://www.google.com/maps/place/%E6%A8%82%E8...,25.008416,121.508817,0.146853,4.1,$1-200,1478,"['土魠魚', '米糕', '排隊', '魷魚', '蝦仁羹', '米粉', '醋', '肉..."
1,鐘點棧 當歸鴨【台北樂華夜市總店】銅板美食|小吃店外送推薦,新北市,保平路18巷8號,https://www.google.com/maps/place/%E9%90%98%E9...,25.007947,121.509835,0.080487,4.2,$1-200,1252,"['碗', '鴨肉飯', '當歸湯', '麵線', '米血', '腿', '香腸', '鴨油..."


In [19]:
df_store = t_clean_tag(df_store)
df_store

,餐廳名稱,nm_city,地址,餐廳連結,latitude,longitude,distance,st_score,st_price,st_total_com,st_tag
0,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,https://www.google.com/maps/place/%E6%A8%82%E8...,25.008416,121.508817,0.146853,4.1,$1-200,1478,"土魠魚, 米糕, 排隊, 魷魚, 蝦仁羹, 米粉, 醋, 肉鬆, 勾芡, 魚塊"
1,鐘點棧 當歸鴨【台北樂華夜市總店】銅板美食|小吃店外送推薦,新北市,保平路18巷8號,https://www.google.com/maps/place/%E9%90%98%E9...,25.007947,121.509835,0.080487,4.2,$1-200,1252,"碗, 鴨肉飯, 當歸湯, 麵線, 米血, 腿, 香腸, 鴨油, 中藥, 地瓜葉"
2,郭記麻辣臭豆腐,新北市,永平路180巷,https://www.google.com/maps/place/%E9%83%AD%E8...,25.009145,121.507948,0.242539,3.8,$1-200,1886,"油飯, 鴨血, 羹, 樂華夜市, 魚乾, 什錦, 香菇, 肚, 四神湯, 麻辣湯"
3,成銘月亮蝦餅,新北市,保平路18巷2號,https://www.google.com/maps/place/%E6%88%90%E9...,25.008341,121.508900,0.139700,4.0,$1-200,384,"蝦子, 炸, 內餡, 樂華夜市, 雞排, 排隊, 酥脆, 泰式, 直播, 椒鹽"
4,顏記深海旗魚串,新北市,永和路一段129號,https://www.google.com/maps/place/%E9%A1%8F%E8...,25.008657,121.511436,0.118925,4.4,$1-200,427,"魚漿, 樂華夜市, 排隊, 水煮蛋, 芥末, 醬油, 鍋, 辣醬, 包裹, 假日"
...,...,...,...,...,...,...,...,...,...,...,...
132,樂華夜市-越南椰子冰沙咖啡,新北市,永和保福路一段27號,https://www.google.com/maps/place/%E6%A8%82%E8...,25.008448,121.509789,0.049435,5.0,$1-200,18,"甜, 椰奶"
133,胖妞仙草奶凍,新北市,永平路28號,https://www.google.com/maps/place/%E8%83%96%E5...,25.008744,121.509568,0.073037,3.4,NaN,77,"檸檬, 價格, 蔓越莓, 綠茶, 樂華夜市"
134,莊家班麻油雞,新北市,保平路28號,https://www.google.com/maps/place/%E8%8E%8A%E5...,25.008682,121.513309,0.307746,3.6,$200-400,359,"麵線, 湯, 碗, 腰子, 雞腿, 冷, 米酒, 夜市, 豬肝, 里肌肉"
135,168魷魚羹,新北市,永平路81巷1號號,https://www.google.com/maps/place/168%E9%AD%B7...,25.008047,121.511846,0.169593,3.7,$1-200,69,"臭豆腐, 麵, 價格, 沙茶, 滷肉飯, 夜市, 肉圓, 蝦仁, 碗, 花枝"


In [20]:
df_comment = e_load_raw_data(nightmarket,"comments")
df_comment.head(2)

,st_name,user_name,user_id,local_guide,rating_stars,time_num,time_unit,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,https://www.google.com/maps/contrib/1030441477...,在地嚮導 · 76 則評論 · 25 張相片,4 顆星,11,小時,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,https://www.google.com/maps/contrib/1112043146...,"在地嚮導 · 2,880 則評論 · 58,024 張相片",4 顆星,1,天,NaN,2025-05-02,2025-05-02


In [21]:
df_comment = t_get_user_id(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,local_guide,rating_stars,time_num,time_unit,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,在地嚮導 · 76 則評論 · 25 張相片,4 顆星,11,小時,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,"在地嚮導 · 2,880 則評論 · 58,024 張相片",4 顆星,1,天,NaN,2025-05-02,2025-05-02


In [22]:
df_comment = t_is_localguide(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,is_local_guide,local_guide,rating_stars,time_num,time_unit,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,TRUE,76 則評論 · 25 張相片,4 顆星,11,小時,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,TRUE,"2,880 則評論 · 58,024 張相片",4 顆星,1,天,NaN,2025-05-02,2025-05-02


In [23]:
df_comment = t_get_comment_count(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,is_local_guide,review_count,local_guide,rating_stars,time_num,time_unit,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,TRUE,76,76 則評論 · 25 張相片,4 顆星,11,小時,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,TRUE,2,"2,880 則評論 · 58,024 張相片",4 顆星,1,天,NaN,2025-05-02,2025-05-02


In [24]:
df_comment = t_get_photo_count(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,is_local_guide,review_count,photo_count,rating_stars,time_num,time_unit,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,TRUE,76,25,4 顆星,11,小時,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,TRUE,2,58,4 顆星,1,天,NaN,2025-05-02,2025-05-02


In [25]:
df_comment = t_get_time(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,is_local_guide,review_count,photo_count,rating_stars,time_num,time_unit,months_ago,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,TRUE,76,25,4 顆星,11,小時,0.000000,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,TRUE,2,58,4 顆星,1,天,0.033333,NaN,2025-05-02,2025-05-02


In [26]:
df_comment = t_get_star(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,is_local_guide,review_count,photo_count,rating_star,time_num,time_unit,months_ago,comment,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,TRUE,76,25,4,11,小時,0.000000,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,TRUE,2,58,4,1,天,0.033333,NaN,2025-05-02,2025-05-02


In [27]:
df_comment = t_clean_comment(df_comment)
df_comment.head(2)

,st_name,user_name,user_id,is_local_guide,review_count,photo_count,rating_star,time_num,time_unit,months_ago,content_clean,create_date,update_date
0,樂華夜市三鮮羹,Pipi Lee,103044147779368367807,TRUE,76,25,4,11,小時,0.000000,新鮮好吃！,2025-05-02,2025-05-02
1,樂華夜市三鮮羹,Tommy,111204314645120534398,TRUE,2,58,4,1,天,0.033333,None,2025-05-02,2025-05-02


In [28]:
df_store.drop(["latitude","longitude","distance"],axis=1,inplace=True)
df_store.insert(0,"nm_name",nightmarket_ch)
df_store.rename(columns={"餐廳名稱":"st_name","地址":"st_address","餐廳連結":"st_url"},inplace=True)
df_store.head(3)

,nm_name,st_name,nm_city,st_address,st_url,st_score,st_price,st_total_com,st_tag
0,樂華夜市,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,https://www.google.com/maps/place/%E6%A8%82%E8...,4.1,$1-200,1478,"土魠魚, 米糕, 排隊, 魷魚, 蝦仁羹, 米粉, 醋, 肉鬆, 勾芡, 魚塊"
1,樂華夜市,鐘點棧 當歸鴨【台北樂華夜市總店】銅板美食|小吃店外送推薦,新北市,保平路18巷8號,https://www.google.com/maps/place/%E9%90%98%E9...,4.2,$1-200,1252,"碗, 鴨肉飯, 當歸湯, 麵線, 米血, 腿, 香腸, 鴨油, 中藥, 地瓜葉"
2,樂華夜市,郭記麻辣臭豆腐,新北市,永平路180巷,https://www.google.com/maps/place/%E9%83%AD%E8...,3.8,$1-200,1886,"油飯, 鴨血, 羹, 樂華夜市, 魚乾, 什錦, 香菇, 肚, 四神湯, 麻辣湯"


In [29]:
df = df_store.merge(df_comment,how="right",left_on="st_name",right_on="st_name")
df.drop(["st_score","st_price","st_total_com","st_tag","st_url"],axis=1,inplace=True)
from datetime import date
df["update_date"] = date.today()
df

,nm_name,st_name,nm_city,st_address,user_name,user_id,is_local_guide,review_count,photo_count,rating_star,time_num,time_unit,months_ago,content_clean,create_date,update_date
0,樂華夜市,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,Pipi Lee,103044147779368367807,TRUE,76,25,4,11,小時,0.000000,新鮮好吃！,2025-05-02,2025-05-15
1,樂華夜市,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,Tommy,111204314645120534398,TRUE,2,58,4,1,天,0.033333,None,2025-05-02,2025-05-15
2,樂華夜市,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,笑笑看,102017796226617730331,TRUE,153,162,4,2,天,0.066667,偏甜的湯頭.越來越貴的價位.但還是很好吃.只是幾乎都要排隊,2025-05-02,2025-05-15
3,樂華夜市,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,Ching Huang,116360913705848267761,TRUE,373,2,5,5,天,0.166667,好吃\r\n不用加麵\r\n份量剛好\r\n排隊很長\r\n吃完很熱\r\n送上食物很快,2025-05-02,2025-05-15
4,樂華夜市,樂華夜市三鮮羹,新北市,永平路146號樂華夜市內,Hsuan,117506998133029465058,TRUE,622,2,3,2,週,0.500000,魷魚羹米粉是甜的，重口味的我不喜。,2025-05-02,2025-05-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77661,樂華夜市,下港排骨酥湯（僅外帶）,新北市,保平路14號,林愛蒂,100044919315694636499,FALSE,2,3,3,13,年,234.000000,None,2025-05-05,2025-05-15
77662,樂華夜市,下港排骨酥湯（僅外帶）,新北市,保平路14號,Marcus Ding,111405490873974325387,FALSE,98,562,1,13,年,234.000000,None,2025-05-05,2025-05-15
77663,樂華夜市,下港排骨酥湯（僅外帶）,新北市,保平路14號,Lo Bear,102959267630318562640,FALSE,10,7,5,13,年,234.000000,None,2025-05-05,2025-05-15
77664,樂華夜市,下港排骨酥湯（僅外帶）,新北市,保平路14號,邱世敏,113326797252223850753,TRUE,8,202,4,13,年,234.000000,None,2025-05-05,2025-05-15


In [30]:
df["months_ago"] = df["months_ago"] + (pd.to_datetime(df["update_date"]) - pd.to_datetime(df["create_date"])).dt.days / 30

In [31]:
df_store.to_csv(f"{nightmarket}_restaurants_Cleaned.csv",index=False,header=True,encoding="utf-8-sig")
df.to_csv(f"{nightmarket}_comment_Cleaned.csv",index=False,header=True,encoding="utf-8'sig")